# Assignment 7 – Fine-Tuning LLM with LoRA

This notebook demonstrates fine-tuning a pre-trained language model using **LoRA (Low-Rank Adaptation)** within the `FinetuningLLM-week07` environment.

## Environment
- Python 3.10.19  
- Torch 2.7.0  
- Transformers 4.57.3  
- Datasets 4.4.1  
- Scikit-learn 1.7.2  

Make sure to update `numexpr` to 2.8.4+ to avoid warnings:
```bash
pip install --upgrade numexpr

# 1. Data Loading
Import and preprocess dataset with HuggingFace `datasets`.

In [1]:
# === Imports Cell ===
import torch
import transformers
import datasets
import sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Scikit-learn:", sklearn.__version__)


Torch: 2.7.0
Transformers: 4.57.3
Datasets: 4.4.1
Scikit-learn: 1.7.2


In [2]:
from datasets import load_dataset

# Load a sample dataset from Hugging Face
# Example: sentiment analysis dataset (binary classification)
dataset = load_dataset("imdb")

# Split into train and eval sets
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))   # small subset for demo
eval_dataset = dataset["test"].shuffle(seed=42).select(range(1000))    # small subset for demo

# Inspect a sample
print(train_dataset[0])

{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'label': 1}


In [3]:
# === Dataset Formatting Cell ===

# Rename "label" column to "labels" for Trainer compatibility
train_dataset = train_dataset.rename_column("label", "labels")
eval_dataset = eval_dataset.rename_column("label", "labels")

# Ensure PyTorch tensors are returned
train_dataset.set_format("torch")
eval_dataset.set_format("torch")

print(train_dataset[0])

{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'labels': tensor(1)}


# 2. Model Setup
Load a base transformer model for fine-tuning.

In [5]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# 1. Pick the base model
model_name = "bert-base-uncased"

# 2. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. Load model with classification head
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 4. Define tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],            # the raw text field from IMDB dataset
        padding="max_length",        # pad to fixed length
        truncation=True,             # cut off if too long
        max_length=256               # typical length for classification
    )

# 5. Apply tokenizer to train and eval sets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# 6. Format for PyTorch
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_eval.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Loaded model: {model_name}")
print(tokenized_train[0])

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded model: bert-base-uncased
{'labels': tensor(1), 'input_ids': tensor([  101,  2045,  2003,  2053,  7189,  2012,  2035,  2090,  3481,  3771,
         1998,  6337,  2099,  2021,  1996,  2755,  2008,  2119,  2024,  2610,
         2186,  2055,  6355,  6997,  1012,  6337,  2099,  3504, 15594,  2100,
         1010,  3481,  3771,  3504,  4438,  1012,  6337,  2099, 14811,  2024,
         3243,  3722,  1012,  3481,  3771,  1005,  1055,  5436,  2024,  2521,
         2062,  8552,  1012,  1012,  1012,  3481,  3771,  3504,  2062,  2066,
         3539,  8343,  1010,  2065,  2057,  2031,  2000,  3962, 12319,  1012,
         1012,  1012,  1996,  2364,  2839,  2003,  5410,  1998,  6881,  2080,
         1010,  2021,  2031,  1000, 17936,  6767,  7054,  3401,  1000,  1012,
         2111,  2066,  2000, 12826,  1010,  2000,  3648,  1010,  2000, 16157,
         1012,  2129,  2055,  2074,  9107,  1029,  6057,  2518,  2205,  1010,
         2111,  3015,  3481,  3771,  3504,  2137,  2021,  1010,  2006,  199

In [7]:
### Sanity check cell
import torch

# Sanity check: run one batch through the model
sample_batch = tokenized_train[:4]  # grab 4 samples
inputs = {
    "input_ids": sample_batch["input_ids"],
    "attention_mask": sample_batch["attention_mask"],
    "labels": sample_batch["labels"]
}

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

print("Logits shape:", outputs.logits.shape)
print("Sample logits:", outputs.logits)

Logits shape: torch.Size([4, 2])
Sample logits: tensor([[-0.2199, -0.4513],
        [-0.2119, -0.3980],
        [-0.1892, -0.5077],
        [-0.2760, -0.4847]])


In [8]:
# Label mapping: 0 = negative, 1 = positive
id2label = {0: "negative", 1: "positive"}

# Example: run one batch through the model and decode predictions
sample_batch = tokenized_train[:4]
inputs = {
    "input_ids": sample_batch["input_ids"],
    "attention_mask": sample_batch["attention_mask"]
}

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
predictions = torch.argmax(logits, dim=-1)

for i, pred in enumerate(predictions):
    text_label = id2label[pred.item()]
    print(f"Sample {i} → Predicted: {text_label} (raw: {pred.item()})")


Sample 0 → Predicted: negative (raw: 0)
Sample 1 → Predicted: negative (raw: 0)
Sample 2 → Predicted: negative (raw: 0)
Sample 3 → Predicted: negative (raw: 0)


# 3. LoRA Configuration
Apply parameter-efficient fine-tuning using LoRA.

In [9]:
from peft import LoraConfig, get_peft_model

# Define LoRA configuration
lora_config = LoraConfig(
    r=8,                      # rank (size of the adapter matrices)
    lora_alpha=32,            # scaling factor
    lora_dropout=0.1,          # dropout for regularization
    bias="none",               # don't train bias terms
    task_type="SEQ_CLS"        # sequence classification task
)

# Wrap the base model with LoRA
model = get_peft_model(model, lora_config)

# Print summary of trainable parameters
model.print_trainable_parameters()

trainable params: 296,450 || all params: 109,780,228 || trainable%: 0.2700


In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, average="binary"),
        "recall": recall_score(labels, predictions, average="binary"),
        "f1": f1_score(labels, predictions, average="binary"),
    }

# 4. Training
Run fine-tuning loop with evaluation checkpoints.

In [11]:
from transformers import TrainingArguments, Trainer

# Training arguments (compatible with Transformers 4.57.3)
training_args = TrainingArguments(
    output_dir="./results",          # where to save checkpoints
    eval_strategy="epoch",           # <-- use eval_strategy instead of evaluation_strategy
    save_strategy="epoch",           # save model each epoch
    learning_rate=2e-4,              # smaller LR since LoRA is efficient
    per_device_train_batch_size=16,  # adjust based on GPU memory
    per_device_eval_batch_size=16,
    num_train_epochs=3,              # tweak as needed
    weight_decay=0.01,               # regularization
    logging_dir="./logs",            # log directory
    logging_steps=50,
    load_best_model_at_end=True
)


# Define compute_metrics using scikit-learn
def compute_metrics(eval_pred):
    import numpy as np
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,   # <-- replace with your dataset variable
    eval_dataset=tokenized_eval,     # <-- replace with your dataset variable
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Start training
trainer.train()

C:\Users\barrc\AppData\Local\Temp\ipykernel_93936\2440391101.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
C:\Users\barrc\anaconda3\envs\FinetuningLLM-week07\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.530700,0.401386,0.816000,0.756757,0.918033,0.829630
2,0.294100,0.408288,0.832000,0.761438,0.954918,0.847273
3,0.260600,0.326523,0.865000,0.833648,0.903689,0.867257


C:\Users\barrc\anaconda3\envs\FinetuningLLM-week07\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\barrc\anaconda3\envs\FinetuningLLM-week07\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=375, training_loss=0.3856171048482259, metrics={'train_runtime': 4590.5766, 'train_samples_per_second': 1.307, 'train_steps_per_second': 0.082, 'total_flos': 792065249280000.0, 'train_loss': 0.3856171048482259, 'epoch': 3.0})

# 5. Evaluation
Use `scikit-learn` metrics (accuracy, precision, recall, F1, confusion matrix).

In [12]:
# Evaluate the fine-tuned model
eval_results = trainer.evaluate()

print("Evaluation Results:")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

C:\Users\barrc\anaconda3\envs\FinetuningLLM-week07\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Evaluation Results:
eval_loss: 0.3265
eval_accuracy: 0.8650
eval_precision: 0.8336
eval_recall: 0.9037
eval_f1: 0.8673
eval_runtime: 156.9447
eval_samples_per_second: 6.3720
eval_steps_per_second: 0.4010
epoch: 3.0000


In [14]:
def predict_sentiment(text):
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt")
    # Run the model
    outputs = model(**inputs)
    # Get the predicted class (0 or 1)
    prediction = outputs.logits.argmax(-1).item()
    # Map to human-readable label
    return "Positive" if prediction == 1 else "Negative"

# Demo examples
print(predict_sentiment("This movie was absolutely fantastic!"))
print(predict_sentiment("The plot was boring and predictable."))
print(predict_sentiment("I loved the acting but the story was weak."))

Positive
Negative
Negative


# 6. Demo
Generate predictions to validate fine-tuning results.

In [15]:
import torch

# 6. Demo: test the fine-tuned model on new text
def predict(text):
    # Tokenize the input
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=256)
    
    # Run through the model
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get predicted class
    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=-1).item()
    
    return predicted_class

# Try a few examples
sample_texts = [
    "I absolutely loved this movie, it was fantastic!",
    "This film was boring and a complete waste of time."
]

for text in sample_texts:
    pred = predict(text)
    print(f"Text: {text}\nPredicted class: {pred}\n")

Text: I absolutely loved this movie, it was fantastic!
Predicted class: 1

Text: This film was boring and a complete waste of time.
Predicted class: 0

